# 01 — Extract Shot Attempts
Load Wyscout event data league-by-league, filter to shot attempts (standard shots,
in-game penalties, and direct free-kick shots), extract coordinates and goal outcomes,
then save combined, train, and test parquet files.

> **"Shot attempts"** in this notebook means: `Shot/Shot`, `Free Kick/Penalty` (matchPeriod ≠ P),
> and `Free Kick/Free kick shot`. Legacy filenames retain "shots" for continuity.

In [1]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

In [2]:
import sys
!{sys.executable} -m pip install pyarrow



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 0: Path validation

In [3]:
DATA_DIR = Path("../data/wyscout-data/events")
assert DATA_DIR.exists(), f"Missing directory: {DATA_DIR}"
assert (DATA_DIR / "events_England.json").exists(), "England events file not found"
print("Paths OK")

Paths OK


## Step 1: Helper functions

In [4]:
def extract_x(pos):
    return pos[0]["x"] if pos else None

def extract_y(pos):
    return pos[0]["y"] if pos else None

def extract_goal(tags):
    return int(any(tag["id"] == 101 for tag in tags))

def is_shot_attempt(e):
    en = e.get("eventName", "")
    sn = e.get("subEventName", "")
    if en == "Shot" and sn == "Shot":
        return True
    if en == "Free Kick" and sn == "Penalty" and e.get("matchPeriod") != "P":
        return True
    if en == "Free Kick" and sn == "Free kick shot":
        return True
    return False

## Step 2: Load shot attempts per league (memory-efficient)
Filter to shot attempts at read time — this includes:
- `Shot / Shot` — standard open-play shots
- `Free Kick / Penalty` — in-game awarded penalties (matchPeriod ≠ "P")
- `Free Kick / Free kick shot` — direct free-kick shots

Penalty shootout kicks (matchPeriod == "P") are excluded — different context, would skew xG estimates.

In [5]:
leagues = ["England", "France", "Germany", "Italy", "Spain"]
KEEP_COLS = [
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod", "eventName", "subEventName",
    "x", "y", "is_goal", "is_penalty", "is_direct_free_kick",
]

events_by_league = {}

for league in leagues:
    with open(DATA_DIR / f"events_{league}.json", encoding="utf-8") as f:
        events = json.load(f)

    attempts = [e for e in events if is_shot_attempt(e)]
    for e in attempts:
        e["league"] = league

    league_df = pd.DataFrame(attempts)
    league_df["x"]       = pd.to_numeric(league_df["positions"].apply(extract_x), errors="coerce")
    league_df["y"]       = pd.to_numeric(league_df["positions"].apply(extract_y), errors="coerce")
    league_df["is_goal"] = league_df["tags"].apply(extract_goal).astype("int8")
    league_df["is_penalty"] = (
        (league_df["eventName"] == "Free Kick") & (league_df["subEventName"] == "Penalty")
    ).astype("int8")
    league_df["is_direct_free_kick"] = (
        (league_df["eventName"] == "Free Kick") & (league_df["subEventName"] == "Free kick shot")
    ).astype("int8")

    events_by_league[league] = league_df[KEEP_COLS]
    print(f"{league}: {len(league_df):,} attempts  "
          f"(penalties: {league_df['is_penalty'].sum()}, "
          f"direct FK shots: {league_df['is_direct_free_kick'].sum()})")

England: 8,881 attempts  (penalties: 80, direct FK shots: 350)
France: 8,977 attempts  (penalties: 129, direct FK shots: 521)
Germany: 7,290 attempts  (penalties: 93, direct FK shots: 299)
Italy: 9,347 attempts  (penalties: 126, direct FK shots: 415)
Spain: 8,545 attempts  (penalties: 113, direct FK shots: 453)


## Step 3: Build train/test DataFrames
Training leagues: France, Germany, Italy, Spain  
Test league: England (held out)

In [6]:
train_df = pd.concat(
    [events_by_league[l] for l in ["France", "Germany", "Italy", "Spain"]],
    ignore_index=True
)
test_df = events_by_league["England"].copy()

print(f"Train shots: {len(train_df):,}")
print(f"Test shots:  {len(test_df):,}")

Train shots: 34,159
Test shots:  8,881


## Step 4: Combine and preview

In [7]:
shots = pd.concat([train_df, test_df], ignore_index=True)
shots[["league", "subEventName", "x", "y", "is_goal"]].head()

,league,subEventName,x,y,is_goal
0,France,Shot,94,57,1
1,France,Shot,83,42,0
2,France,Shot,96,43,1
3,France,Shot,84,21,0
4,France,Shot,73,51,0


## Step 5: Sanity checks

> **Note:** This notebook extracts shot-level event data only.  
> Match dates and chronological ordering will be added in the next notebook using the matches files.

In [8]:
print("Missing coordinates:")
print(shots[["x", "y"]].isna().sum())

Missing coordinates:
x    0
y    0
dtype: int64


In [9]:
print("Coordinate ranges:")
print(f"x: {shots['x'].min()} to {shots['x'].max()}")
print(f"y: {shots['y'].min()} to {shots['y'].max()}")

Coordinate ranges:
x: 1 to 100
y: 0 to 100


In [10]:
print(shots[["x", "y", "is_goal"]].dtypes)

x          int64
y          int64
is_goal     int8
dtype: object


In [11]:
# Hard schema assertion: exactly these (eventName, subEventName) pairs are allowed
ALLOWED_PAIRS = {
    ("Shot",      "Shot"),
    ("Free Kick", "Penalty"),
    ("Free Kick", "Free kick shot"),
}
actual_pairs = set(zip(shots["eventName"], shots["subEventName"]))
assert actual_pairs == ALLOWED_PAIRS, (
    f"Extra pairs: {actual_pairs - ALLOWED_PAIRS} | Missing pairs: {ALLOWED_PAIRS - actual_pairs}"
)

# Count sanity: both new event types must be present
assert (shots["is_penalty"] == 1).sum() > 0, "No penalties found — check extraction filter"
assert (shots["is_direct_free_kick"] == 1).sum() > 0, "No direct FK shots found — check extraction filter"

# Mutual exclusivity
assert not ((shots["is_penalty"] == 1) & (shots["is_direct_free_kick"] == 1)).any(), \
    "A row has both is_penalty and is_direct_free_kick set to 1"

# Penalty shootout guard
assert not ((shots["is_penalty"] == 1) & (shots["matchPeriod"] == "P")).any(), \
    "Penalty shootout kick found — check matchPeriod filter"

print("Event-type counts:")
print(shots.groupby(["eventName", "subEventName"]).size().to_string())
print(f"\nTotal attempts: {len(shots):,}")
print(f"  Standard shots:   {((shots['is_penalty'] == 0) & (shots['is_direct_free_kick'] == 0)).sum():,}")
print(f"  Penalties:        {(shots['is_penalty'] == 1).sum():,}")
print(f"  Direct FK shots:  {(shots['is_direct_free_kick'] == 1).sum():,}")

Event-type counts:
eventName  subEventName  
Free Kick  Free kick shot     2038
           Penalty             541
Shot       Shot              40461

Total attempts: 43,040
  Standard shots:   40,461
  Penalties:        541
  Direct FK shots:  2,038


In [12]:
print("Duplicate rows:", shots.duplicated().sum())

Duplicate rows: 0


In [13]:
print(f"Total attempts: {len(shots):,}")
print(f"Goals:          {shots['is_goal'].sum():,}")
print(f"Goal rate:      {shots['is_goal'].mean():.2%}")

league_summary = shots.groupby("league").agg(
    attempts=("is_goal", "count"),
    penalties=("is_penalty", "sum"),
    direct_fk_shots=("is_direct_free_kick", "sum"),
    goals=("is_goal", "sum"),
    goal_rate=("is_goal", "mean"),
)
print("\nLeague-level summary:")
display(league_summary)

Total attempts: 43,040
Goals:          4,790
Goal rate:      11.13%

League-level summary:


,attempts,penalties,direct_fk_shots,goals,goal_rate
league,,,,,
England,8881,80,350,988,0.111249
France,8977,129,521,998,0.111173
Germany,7290,93,299,833,0.114266
Italy,9347,126,415,978,0.104633
Spain,8545,113,453,993,0.116208


## Step 6: Save outputs

In [14]:
OUT = Path("../data")
OUT.mkdir(parents=True, exist_ok=True)

shots.to_parquet(OUT / "wyscout_shots.parquet", index=False)
train_df.to_parquet(OUT / "wyscout_train_shots.parquet", index=False)
test_df.to_parquet(OUT / "wyscout_test_shots.parquet", index=False)
shots.head(100).to_csv(OUT / "wyscout_shots_sample.csv", index=False)

print(f"Saved {len(shots):,} combined attempts  → wyscout_shots.parquet")
print(f"Saved {len(train_df):,} train attempts   → wyscout_train_shots.parquet")
print(f"Saved {len(test_df):,} test attempts    → wyscout_test_shots.parquet")
print("Saved 100-row sample                   → wyscout_shots_sample.csv")
print(f"\nColumns: {list(shots.columns)}")

Saved 43,040 combined attempts  → wyscout_shots.parquet
Saved 34,159 train attempts   → wyscout_train_shots.parquet
Saved 8,881 test attempts    → wyscout_test_shots.parquet
Saved 100-row sample                   → wyscout_shots_sample.csv

Columns: ['league', 'matchId', 'playerId', 'teamId', 'eventSec', 'matchPeriod', 'eventName', 'subEventName', 'x', 'y', 'is_goal', 'is_penalty', 'is_direct_free_kick']


In [15]:
check_df = pd.read_parquet(OUT / "wyscout_shots.parquet")
assert check_df.shape[0] == len(shots), "Row count mismatch on reload"
expected_cols = set(KEEP_COLS)
assert expected_cols.issubset(check_df.columns), f"Missing columns: {expected_cols - set(check_df.columns)}"
assert "positions" not in check_df.columns, "positions column should not be in saved parquet"
assert "tags" not in check_df.columns, "tags column should not be in saved parquet"
print(check_df.shape)
check_df.head()

(43040, 13)


,league,matchId,playerId,teamId,eventSec,matchPeriod,eventName,subEventName,x,y,is_goal,is_penalty,is_direct_free_kick
0,France,2500686,256992,3799,605.975493,1H,Shot,Shot,94,57,1,0,0
1,France,2500686,334552,3772,859.236394,1H,Shot,Shot,83,42,0,0,0
2,France,2500686,26389,3772,1568.104834,1H,Shot,Shot,96,43,1,0,0
3,France,2500686,276920,3772,1800.852078,1H,Shot,Shot,84,21,0,0,0
4,France,2500686,366760,3799,2009.537139,1H,Shot,Shot,73,51,0,0,0
